In [1]:
# Parameters
# (do not containerize this cell)
# =====
# Input
# -----
# user define/upload
# Opt1, file
# param_file_seagrass_site_data = "[Seagrass_site_data.xlsx]"
param_file_seagrass_site_data = "Seagrass_site_data.xlsx"

# Opt2, GUI
# param_latitude <- 56.0953
# param_longitude <- 14.2785
# param_seagrass_species <- "Posidonia oceanica"
# # param_seagrass_species <- list("Cymodocea nodosa", "Halophila stipulacea", "Posidonia oceanica", "Zostera marina", "Zostera noltei", "Zostera marina and Cymodocea nodosa", "Zostera marina and Zostera noltei")

# exist/download
# =====
# Default
# Used in step01
conf_file_bottomT_p95               <- "bottomT_p95_daily_C.nc"
conf_file_uo_mean_1.5m_m_s          <- "uo_mean_1.5m_m_s.nc"
conf_file_vo_p90_1.5m_m_s           <- "vo_p90_1.5m_m_s.nc"
conf_file_po4_mean_1.5m_mmol_m3     <- "po4_mean_monthly_1.5m_mmol_m3.nc"
conf_file_pH_mean_1.5m              <- "pH_mean_monthly_1.5m.nc"
conf_file_wave_height_VHM0_p95_m    <- "wave_height_p95_m.nc"
conf_file_Surf_fgco2_p95_molC_m2_yr <- "Surf_fgco2_p95_molC_m2_yr.nc"
conf_file_KD                        <- "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc"
conf_file_RRS443                    <- "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km.nc"
# Used in step02
conf_file_For_modeling_df_shallow_carbon_density <- "For_modeling_df_shallow_carbon_density.rds"
conf_file_GAM_top_reduced_SGstock   <- "GAM_top_reduced_model_SGstock.rds"

# Output
# -----
# results
conf_file_result_txt <- "Seagrass_site_result.txt"
conf_file_result_csv <- "Seagrass_site_result.csv"

# User
# -----
# param_user_email = "[the e-mail address you used to log into NaaVRE]"
# param_user_email = "quan.pan@lifewatch.eu"

# user define/upload, 1: dummy data; 0: to define/upload
conf_use_dummy_data <- 1
# # exist/download, 1: existing/downloaded data; 0: to download
# param_use_exist_data <- 1

# Temporary
# -----
# dir
conf_tmp_directory_data <- "/tmp/data"
# file, exchange data between cells
conf_tmp_file_SG_modeling_dataframe <- "SG_modeling_dataframe.csv"
conf_tmp_file_result <- "tmp_result.rds"

# MINIO
# =====
# # old version, need token input
# # -----
# # /home/jovyan/Virtual Labs/Seagrass carbon stock model/Git public
# conf_minio_endpoint <- "scruffy.lab.uvalight.net:9000"
# conf_minio_region   <- "nl-uvalight"
# conf_minio_public_bucket <- "naa-vre-public"
# conf_minio_user_bucket   <- "naa-vre-user-data"
# conf_minio_public_bucket_root <- "vl-bluecarbon"
# # Secrets (do not containerize this cell)
# # -----
# library("SecretsProvider")
# 
# secretsProvider <- SecretsProvider()
# secret_minio_access_key = ""
# secret_minio_access_key = secretsProvider$get_secret("secret_minio_access_key")
# secret_minio_secret_key = ""
# secret_minio_secret_key = secretsProvider$get_secret("secret_minio_secret_key")
# # MinIO data retriever
# # -----
# library("aws.s3")
# 
# Sys.setenv(
#     "AWS_S3_ENDPOINT"    = conf_minio_endpoint,
#     "AWS_DEFAULT_REGION" = conf_minio_region,
#     "AWS_ACCESS_KEY_ID"     = secret_minio_access_key,
#     "AWS_SECRET_ACCESS_KEY" = secret_minio_secret_key
# )
# aws.s3::save_object(bucket=conf_minio_public_bucket, object=file_path, file=file_bottomT_p95)
# minio_status <- put_object(bucket=conf_minio_user_bucket, file=file_result_csv, object=paste(param_user_email, fcloud_result_csv, sep="/"))

# new version, no need token input
# -----
# old: file_path <- paste(
#     param_user_email,
#     param_file,
#     sep="/")
# new: file_path <- paste(
#     conf_minio_bucket_local_path,
#     conf_minio_public_bucket,
#     conf_minio_public_bucket_root,
#     fname,
#     sep="/")
# file_path <- paste(
#     conf_minio_bucket_local_path,
#     conf_minio_user_bucket,
#     fname,
#     sep="/")
conf_minio_public_bucket      <- "naa-vre-public"
conf_minio_public_bucket_root <- "vl-bluecarbon"
conf_minio_user_bucket        <- "naa-vre-user-data"
# conf_minio_user_bucket_root   <- param_user_email
conf_minio_user_local_root    <- "/home/jovyan/Cloud Storage"

In [ ]:
# # Test cell, do not cotainized
# file_seagrass_site_data <- paste(conf_minio_user_local_root, conf_minio_user_bucket, param_file_seagrass_site_data, sep="/")

# # Open datafile with seagrass site data
# Seagrass_site <- readxl::read_excel(
#     file_seagrass_site_data,
#     sheet = "Data", 
#     col_types = c("numeric", "numeric", "text")
# )
# Seagrass_site

In [3]:
# Input from Excel

# Prepare
# =====
# Load dependencies
library(tidyverse)
library(RNetCDF)

library(mgcv)
library(dplyr)
library(stringr)

# Ensure the temporary data storage directory exists
dir.create(conf_tmp_directory_data, showWarnings = FALSE)

# # Parameter validator, GUI
# # =====
# seagrass_species_options <- list("Cymodocea nodosa", "Halophila stipulacea", "Posidonia oceanica", "Zostera marina", "Zostera noltei", "Zostera marina and Cymodocea nodosa", "Zostera marina and Zostera noltei")

# validate_seagrass_species <- function(tmp_seagrass_species, options) {
#   is_valid_option <- tmp_seagrass_species %in% options
#   if (tmp_seagrass_species == "") {
#     message("Seagrass species is unspecified")
#     }
#     else if (!is_valid_option) {
#     stop(
#       tmp_seagrass_species,
#       "Is not part of the options of allowed seagrass species: ", 
#       paste(options, collapse = ", ")
#     )
#   } else {
#     message(tmp_seagrass_species, "is a valid the seagrass species.")
#   }
# }

# validate_geographic_coordinate <- function(coordinate_double) {  
#   if (!typeof(coordinate_double) == "double") {
#       stop(
#           "The coordinate: ", coordinate_double, " is not a double (e.g. 56.0953, not `56.0953`)"
#       )
#   }
#   # coordinate_string <- as.character(coordinate_double)
#   # pattern <- "^\d{2}\.\d{4}$"
#   # is_valid_format <- grepl(pattern, coordinate_string)
#   # if (!(is_valid_format)) {
#   #     stop(
#   #         "The coordinate: ", coordinate_double, " does not follow the required format: ", pattern, " e.g. 56.0953"
#   #     )
#   # } else {
#   #     message(coordinate_double, " is valid geographic coordinate input.")
#   # }
# }

# validate_seagrass_species(param_seagrass_species, seagrass_species_options)
# validate_geographic_coordinate(param_latitude)
# validate_geographic_coordinate(param_longitude)

# validation_completed <- 1

# Cell output
# =====
file_tmp_SG_modeling_dataframe <- paste(conf_tmp_directory_data, conf_tmp_file_SG_modeling_dataframe, sep="/")
file_tmp_result                <- paste(conf_tmp_directory_data, conf_tmp_file_result, sep="/")

# #####
# Seagrass site data
print("Seagrass site data")

# Download file from bucket S3
# -----
# MINIO old
# file_seagrass_site_data <- paste(conf_tmp_directory_data, param_file_seagrass_site_data, sep="/")
# 
# if (conf_use_dummy_data) {
# file_path <- paste(param_user_email, param_file_seagrass_site_data, sep="/")
# print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
# aws.s3::save_object(bucket=conf_minio_user_bucket, object=file_path, file=file_seagrass_site_data)

# MINIO new
# if (conf_use_dummy_data) {
file_seagrass_site_data <- paste(conf_minio_user_local_root, conf_minio_user_bucket, param_file_seagrass_site_data, sep="/")

# Open datafile with seagrass site data
Seagrass_site <- readxl::read_excel(
    file_seagrass_site_data,
    sheet = "Data", 
    col_types = c("numeric", "numeric", "text")
)

# # GUI input
# # -----
# Seagrass_site <- data.frame(
#     latitude=as.vector(as.numeric(param_latitude)),
#     longitude=as.vector(as.numeric(param_longitude)),
#     seagrass_species=as.vector(unlist(param_seagrass_species)))
# Seagrass_site

# # Convert seagrass_species to a factor variable
# Seagrass_site$seagrass_species <- as.factor(Seagrass_site$seagrass_species)

# #####
# Bottom_T_p95
print("Bottom_T_p95")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_bottomT_p95 <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_bottomT_p95, sep="/")
    } else {
    file_bottomT_p95 <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_bottomT_p95, sep="/")
}

# Open netcdf
bottomT_p95 <- open.nc(file_bottomT_p95)
# print.nc(bottomT_p95)

# Get data
bottomT_p95.value <- var.get.nc(bottomT_p95, "p95_bottomT_daily_C", unpack=TRUE)
bottomT_p95.lat <- var.get.nc(bottomT_p95, "latitude")
bottomT_p95.lon <- var.get.nc(bottomT_p95, "longitude")

# Close netcdf
close.nc(bottomT_p95)

# Start analysis
# -----
# print("Extract closest matching value based on location")
# This first function (extract_values) extracts values of matching sites, however this gives NAs for sites
# that appear over land given that the data product is at 0.083deg
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(bottomT_p95.lat - lat_value))
  lon_index <- which.min(abs(bottomT_p95.lon - lon_value))
  return(bottomT_p95.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site %>%
  rowwise() %>%
  mutate(bottomT_p95_C = extract_values(latitude, longitude))

# print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in bottomT_p95.value
  valid_indices <- which(!is.na(bottomT_p95.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- bottomT_p95.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- bottomT_p95.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value from bottomT_p95.value
  value_at_closest <- bottomT_p95.value[closest_valid_index[1], closest_valid_index[2]]
  
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(bottomT_p95_C_closest = extract_closest_values(latitude, longitude))

# #####
# Uo_mean
print("Uo_mean")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_uo_mean_1.5m_m_s <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_uo_mean_1.5m_m_s, sep="/")
    } else {
    file_uo_mean_1.5m_m_s <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_uo_mean_1.5m_m_s, sep="/")
}

# Open netcdf
uo_mean_1.5m_m_s <- open.nc(file_uo_mean_1.5m_m_s)
# print.nc(uo_mean_1.5m_m_s)

# Get data
uo_mean_1.5m.value <- var.get.nc(uo_mean_1.5m_m_s, 'uo_mean_1.5m_m_s', unpack=TRUE)
uo_mean_1.5m.lat <- var.get.nc(uo_mean_1.5m_m_s, 'latitude')
uo_mean_1.5m.lon <- var.get.nc(uo_mean_1.5m_m_s, 'longitude')

# Close netcdf
close.nc(uo_mean_1.5m_m_s)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(uo_mean_1.5m.lat - lat_value))
  lon_index <- which.min(abs(uo_mean_1.5m.lon - lon_value))
  return(uo_mean_1.5m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(uo_mean_1.5m_m_s = extract_values(latitude, longitude))

# print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in uo_mean_1.5m.value
  valid_indices <- which(!is.na(uo_mean_1.5m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- uo_mean_1.5m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- uo_mean_1.5m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- uo_mean_1.5m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(uo_mean_1.5m_m_s_closest = extract_closest_values(latitude, longitude))

# #####
# Vo_p90
print("Vo_p90")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_vo_p90_1.5m_m_s <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_vo_p90_1.5m_m_s, sep="/")
    } else {
    file_vo_p90_1.5m_m_s <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_vo_p90_1.5m_m_s, sep="/")
}

# Open netcdf
# -----
vo_p90_1.5m_m_s <- open.nc(file_vo_p90_1.5m_m_s)
# print.nc(vo_p90_1.5m_m_s)

# Get data
vo_p90_1.5m.value <- var.get.nc(vo_p90_1.5m_m_s, 'vo_p90_1.5m_m_s', unpack=TRUE)
vo_p90_1.5m.lat <- var.get.nc(vo_p90_1.5m_m_s, 'latitude')
vo_p90_1.5m.lon <- var.get.nc(vo_p90_1.5m_m_s, 'longitude')

# Close netcdf
close.nc(vo_p90_1.5m_m_s)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(vo_p90_1.5m.lat - lat_value))
  lon_index <- which.min(abs(vo_p90_1.5m.lon - lon_value))
  return(vo_p90_1.5m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(vo_p90_1.5m_m_s = extract_values(latitude, longitude))

# print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in vo_p90_1.5m.value
  valid_indices <- which(!is.na(vo_p90_1.5m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- vo_p90_1.5m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- vo_p90_1.5m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- vo_p90_1.5m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(vo_p90_1.5m_m_s_closest = extract_closest_values(latitude, longitude))

# #####
# Phosphate_mean
print("Phosphate_mean")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_po4_mean_1.5m_mmol_m3 <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_po4_mean_1.5m_mmol_m3, sep="/")
    } else {
    file_po4_mean_1.5m_mmol_m3 <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_po4_mean_1.5m_mmol_m3, sep="/")
}

# Open netcdf
# -----
po4_mean_1.5m_mmol_m3 <- open.nc(file_po4_mean_1.5m_mmol_m3)
# print.nc(po4_mean_1.5m_mmol_m3)

# Get data
po4_mean.value <- var.get.nc(po4_mean_1.5m_mmol_m3, 'po4_mean_1.5m_mmol_m3', unpack=TRUE)
po4_mean.lat <- var.get.nc(po4_mean_1.5m_mmol_m3, 'latitude')
po4_mean.lon <- var.get.nc(po4_mean_1.5m_mmol_m3, 'longitude')

# Close netcdf
close.nc(po4_mean_1.5m_mmol_m3) #close file

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(po4_mean.lat - lat_value))
  lon_index <- which.min(abs(po4_mean.lon - lon_value))
  return(po4_mean.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(po4_mean_1.5m_mmol_m3 = extract_values(latitude, longitude))

# print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in po4_mean.value
  valid_indices <- which(!is.na(po4_mean.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- po4_mean.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- po4_mean.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- po4_mean.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(po4_mean_1.5m_mmol_m3_closest = extract_closest_values(latitude, longitude))

# #####
# pH_mean
print("pH_mean")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_pH_mean_1.5m <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_pH_mean_1.5m, sep="/")
    } else {
    file_pH_mean_1.5m <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_pH_mean_1.5m, sep="/")
}

# Open netcdf
# -----
pH_mean_1.5m <- open.nc(file_pH_mean_1.5m)
# print.nc(pH_mean_1.5m)

# Get data
pH_mean.value <- var.get.nc(pH_mean_1.5m, 'pH_mean_1.5m', unpack=TRUE)
pH_mean.lat <- var.get.nc(pH_mean_1.5m, 'latitude')
pH_mean.lon <- var.get.nc(pH_mean_1.5m, 'longitude')

# Close netcdf
close.nc(pH_mean_1.5m)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(pH_mean.lat - lat_value))
  lon_index <- which.min(abs(pH_mean.lon - lon_value))
  return(pH_mean.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(pH_mean_1.5m = extract_values(latitude, longitude))

# print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in pH_mean.value
  valid_indices <- which(!is.na(pH_mean.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- pH_mean.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- pH_mean.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- pH_mean.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(pH_mean_1.5m_closest = extract_closest_values(latitude, longitude))

# #####
# VHM0_p95
print("VHM0_p95")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_wave_height_VHM0_p95_m <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_wave_height_VHM0_p95_m, sep="/")
    } else {
    file_wave_height_VHM0_p95_m <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_wave_height_VHM0_p95_m, sep="/")
}

# Open netcdf
# -----
wave_height_VHM0_p95_m <- open.nc(file_wave_height_VHM0_p95_m)
# print.nc(wave_height_VHM0_p95_m)

# Get data
VHM0_p95_m.value <- var.get.nc(wave_height_VHM0_p95_m, 'wave_height_VHM0_p95_m', unpack=TRUE)
VHM0_p95_m.lat <- var.get.nc(wave_height_VHM0_p95_m, 'latitude')
VHM0_p95_m.lon <- var.get.nc(wave_height_VHM0_p95_m, 'longitude')

# Close netcdf
close.nc(wave_height_VHM0_p95_m)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(VHM0_p95_m.lat - lat_value))
  lon_index <- which.min(abs(VHM0_p95_m.lon - lon_value))
  return(VHM0_p95_m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(wave_height_VHM0_p95_m = extract_values(latitude, longitude))

# print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in VHM0_p95_m.value
  valid_indices <- which(!is.na(VHM0_p95_m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- VHM0_p95_m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- VHM0_p95_m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- VHM0_p95_m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(wave_height_VHM0_p95_m_closest = extract_closest_values(latitude, longitude))

# #####
# fgCO2_p95
print("fgCO2_p95")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_Surf_fgco2_p95_molC_m2_yr <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_Surf_fgco2_p95_molC_m2_yr, sep="/")
    } else {
    file_Surf_fgco2_p95_molC_m2_yr <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_Surf_fgco2_p95_molC_m2_yr, sep="/")
}

# Open netcdf
# -----
Surf_fgco2_p95_molC_m2_yr <- open.nc(file_Surf_fgco2_p95_molC_m2_yr)
# print.nc(Surf_fgco2_p95_molC_m2_yr)

# Get data
Surf_fgco2_p95.value <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'Surf_fgco2_p95_molC_m2_yr', unpack=TRUE)
Surf_fgco2_p95.lat <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'latitude')
Surf_fgco2_p95.lon <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'longitude')

# Close netcdf
close.nc(Surf_fgco2_p95_molC_m2_yr)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(Surf_fgco2_p95.lat - lat_value))
  lon_index <- which.min(abs(Surf_fgco2_p95.lon - lon_value))
  return(Surf_fgco2_p95.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(Surf_fgco2_p95_molC_m2_yr = extract_values(latitude, longitude))

# print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in Surf_fgco2_p95.value
  valid_indices <- which(!is.na(Surf_fgco2_p95.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- Surf_fgco2_p95.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- Surf_fgco2_p95.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- Surf_fgco2_p95.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(Surf_fgco2_p95_molC_m2_yr_closest = extract_closest_values(latitude, longitude))

# #####
# KD490
print("KD490")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_KD <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_KD, sep="/")
    } else {
    file_KD <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_KD, sep="/")
}

# Open netcdf
# -----
KD <- open.nc(file_KD)
# print.nc(KD)

# Get data
KD.Kd490 <- var.get.nc(KD, 'Kd490', unpack=TRUE)
KD.lat <- var.get.nc(KD, 'lat')
KD.lon <- var.get.nc(KD, 'lon')

# Close netcdf
close.nc(KD)

# Start analysis
# -----
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(KD.lat - lat_value))
  lon_index <- which.min(abs(KD.lon - lon_value))
  return(KD.Kd490[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(KD = extract_values(latitude, longitude))

# print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in KD.Kd490
  valid_indices <- which(!is.na(KD.Kd490), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- KD.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- KD.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value from KD.Kd490
  value_at_closest <- KD.Kd490[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(KD_closest = extract_closest_values(latitude, longitude))

# #####
# RRS443
print("RRS443")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_RRS443 <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_RRS443, sep="/")
    } else {
    file_RRS443 <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_RRS443, sep="/")
}

# Open netcdf
# -----
RRS443 <- open.nc(file_RRS443)
# print.nc(RRS443)

# Get data
RRS443.Rrs_443 <- var.get.nc(RRS443, 'Rrs_443', unpack=TRUE)
RRS443.lat <- var.get.nc(RRS443, 'lat')
RRS443.lon <- var.get.nc(RRS443, 'lon')

# Close netcdf
close.nc(RRS443)

# Start analysis
# print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(RRS443.lat - lat_value))
  lon_index <- which.min(abs(RRS443.lon - lon_value))
  return(RRS443.Rrs_443[lon_index, lat_index])  # Adjust index order if needed
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(RRS443 = extract_values(latitude, longitude))

# print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in RRS443.Rrs_443
  valid_indices <- which(!is.na(RRS443.Rrs_443), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- RRS443.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- RRS443.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- RRS443.Rrs_443[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(RRS443_closest = extract_closest_values(latitude, longitude))

# #####
# Remove extra columns
# (should only retain the env. covariates with "_closest")
print("Remove extra columns")

# Select data to save
# -----
Seagrass_site_withEnv <- Seagrass_site_withEnv %>% 
  select(-c("bottomT_p95_C", 
            "uo_mean_1.5m_m_s",
            "vo_p90_1.5m_m_s", 
            "po4_mean_1.5m_mmol_m3", 
            "pH_mean_1.5m", 
            "wave_height_VHM0_p95_m",
            "Surf_fgco2_p95_molC_m2_yr", 
            "KD", 
            "RRS443"))

# Some of the remote sensing variables can still have negative values even though they should not. 
# Set these to zero if this is the case.
Seagrass_site_withEnv$KD_closest[Seagrass_site_withEnv$KD_closest < 0] <- 0
Seagrass_site_withEnv$RRS443_closest[Seagrass_site_withEnv$RRS443_closest < 0] <- 0

# Add a variable for sediment mean depth
# For each row in the current dataframe, expand to 10 rows (with the same values), and add a new column called
# "sediment_mean_depth_cm" with the following entries "5, 15, 25, 35, 45, 55, 65, 75, 85, 95"
depths <- c(5, 15, 25, 35, 45, 55, 65, 75, 85, 95)
Seagrass_site_expanded <- Seagrass_site_withEnv[rep(1:nrow(Seagrass_site_withEnv), each = length(depths)), ]
Seagrass_site_expanded$sediment_mean_depth_cm <- rep(depths, times = nrow(Seagrass_site_withEnv))

# Prepare dataframe
SG_modeling_dataframe <- Seagrass_site_expanded %>%
  mutate(seagrass_species = str_trim(seagrass_species),  # remove leading/trailing spaces
         seagrass_species = str_squish(seagrass_species))  # remove extra internal spaces

# SG_modeling_dataframe$seagrass_species <- as.factor(SG_modeling_dataframe$seagrass_species)
SG_modeling_dataframe$sediment_mean_depth_cm <- as.numeric(SG_modeling_dataframe$sediment_mean_depth_cm)

# Save dataframe to use for model prediction
# -----
# write.csv(SG_modeling_dataframe, file_tmp_SG_modeling_dataframe, row.names = FALSE)
saveRDS(SG_modeling_dataframe, file_tmp_result)

# Cell output
# =====
file_Seagrass_site_expanded <- file_tmp_result

print("EVs loaded")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse


This is mgcv 1.9-4. For overview type '?mgcv'.



[1] "Seagrass site data"


Warning message:
“Coercing text to numeric in A2 / R2C1: '55.292436'”
Warning message:
“Coercing text to numeric in B2 / R2C2: '10.790408'”
Warning message:
“Coercing text to numeric in A3 / R3C1: '51.602651'”
Warning message:
“Coercing text to numeric in B3 / R3C2: '4.119084'”
Warning message:
“Coercing text to numeric in A4 / R4C1: '44.651162'”
Warning message:
“Coercing text to numeric in B4 / R4C2: '-1.09674'”
Warning message:
“Coercing text to numeric in A5 / R5C1: '56.0953'”
Warning message:
“Coercing text to numeric in B5 / R5C2: '14.7202'”
Warning message:
“Coercing text to numeric in A6 / R6C1: '55.4575'”
Warning message:
“Coercing text to numeric in B6 / R6C2: '14.2785'”
Warning message:
“Coercing text to numeric in A7 / R7C1: '55.4215'”
Warning message:
“Coercing text to numeric in B7 / R7C2: '13.8463'”
Warning message:
“Coercing text to numeric in A8 / R8C1: '55.3959'”
Warning message:
“Coercing text to numeric in B8 / R8C2: '12.9828'”
Warning message:
“Coercing text to num

[1] "Bottom_T_p95"
[1] "Uo_mean"
[1] "Vo_p90"
[1] "Phosphate_mean"
[1] "pH_mean"
[1] "VHM0_p95"
[1] "fgCO2_p95"
[1] "KD490"
[1] "RRS443"
[1] "Extract closest matching value based on location"
[1] "Only select non-NA points"
[1] "Remove extra columns"
[1] "EVs loaded"


In [5]:
# Carbon Stock Model from Excel 

# Cell input
# =====
file_tmp_result <- file_Seagrass_site_expanded

# Cell exec
# =====
SG_modeling_dataframe <- readRDS(file_tmp_result)

# #####
# Carbon Density
print("Carbon Density")

# Download file from bucket S3
# -----
if (conf_use_dummy_data) {
    file_For_modeling_df_shallow_carbon_density <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_For_modeling_df_shallow_carbon_density, sep="/")
    } else {
    file_For_modeling_df_shallow_carbon_density <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_For_modeling_df_shallow_carbon_density, sep="/")
}

# Load
For_modeling_df_shallow_carbon_density <- read_rds(file_For_modeling_df_shallow_carbon_density)
str(For_modeling_df_shallow_carbon_density$random_core_variable)  # 382 levels (from 461 cores in full dataset, down to 382 unique cores)

# Check sepcies
"Zostera marina and Zostera noltei" %in% levels(For_modeling_df_shallow_carbon_density$seagrass_species)
setdiff(unique(SG_modeling_dataframe$seagrass_species), levels(For_modeling_df_shallow_carbon_density$seagrass_species))

# From SG_modeling_dataframe
raw_SG <- charToRaw(as.character(SG_modeling_dataframe$seagrass_species[SG_modeling_dataframe$seagrass_species == "Zostera marina/Zostera noltei"][1]))

# From For_modeling_df_shallow_carbon_density
raw_training <- charToRaw(as.character(For_modeling_df_shallow_carbon_density$seagrass_species[For_modeling_df_shallow_carbon_density$seagrass_species == "Zostera marina/Zostera noltei"][1]))
identical(raw_SG, raw_training)

# Match factor levels for seagrass_species
SG_modeling_dataframe$seagrass_species <- factor(
  SG_modeling_dataframe$seagrass_species,
  levels = levels(For_modeling_df_shallow_carbon_density$seagrass_species)
)

# #####
# GAM_top_reduced_SGstock
print("GAM_top_reduced_SGstock")

# Download file from bucket S3
# -----
# Use an existing placeholder level from the training data for random_core_variable
# Doesn't matter which one you use because this will be excluded in the model prediction, so I have just picked one
if (conf_use_dummy_data) {
    file_GAM_top_reduced_SGstock <- paste(conf_minio_user_local_root, conf_minio_public_bucket, conf_minio_public_bucket_root, conf_file_GAM_top_reduced_SGstock, sep="/")
    } else {
    file_GAM_top_reduced_SGstock <- paste(conf_minio_user_local_root, conf_minio_user_bucket, conf_file_GAM_top_reduced_SGstock, sep="/")
}

# Load
# file_tmp_data <- paste(conf_tmp_directory_data, "tmp_data.rds", sep="/")
# For_modeling_df_shallow_carbon_density <- readRDS(file_tmp_data)

SG_modeling_dataframe$random_core_variable <- 
  factor(rep("Baltic Sea_Furumon_NA_2021_NA_NA_56.0953_14.7202_1.5", nrow(SG_modeling_dataframe)), 
         levels = levels(For_modeling_df_shallow_carbon_density$random_core_variable))

# Open carbon density prediction model (prepared in script: SG_carbonstock_model.R)
GAM_top_reduced_SGstock <- readRDS(file_GAM_top_reduced_SGstock)

# Predict carbon density for new samples based on input variables in the "SG_modeling_dataframe.csv" spreadsheet
predicted_carbon_density <- predict(GAM_top_reduced_SGstock,
                                    newdata = SG_modeling_dataframe, 
                                    type = "response",
                                    exclude = "s(random_core_variable)")
# predicted_carbon_density_coreID_notexcluded <- predict(GAM_top_reduced_SGstock, newdata = SG_modeling_dataframe, type = "response")
# predicted_carbon_density_nocoreID <- predict(GAM_top_reduced_SGstock_nocoreID, newdata = SG_modeling_dataframe, type = "response")

# Add predictions to dataframe
SG_modeling_dataframe$predicted_carbon_density <- predicted_carbon_density

# #####
# Carbon stock in the upper 30 and 100 cm
print("Carbon stock in the upper 30 and 100 cm")

# =====
# Calculate carbon stock per 10 cm
# Carbon stock (MgC/ha)=Carbon density (gC/cm³) × Depth (cm) × 100,000,000 (cm2/ha) ÷ 1,000,000 (g/Mg)
# Where:
#   1 hectare = 10,000 m² = 100,000,000 cm²
#   1 Mg = 1,000,000 g
SG_modeling_dataframe$predicted_carbon_stock_per_10cm = 
  SG_modeling_dataframe$predicted_carbon_density*10*100000000*(1/1000000)

# Create sample_ID column
SG_modeling_dataframe$sample_ID <- rep(1:(nrow(SG_modeling_dataframe)/10), each = 10)

# Calculate carbon stock in upper 30 and 100 cm 
SG_modeling_dataframe <- SG_modeling_dataframe %>%
  group_by(sample_ID) %>%
  mutate(
    carbon_stock_Mg_ha_upper30cm  = sum(predicted_carbon_stock_per_10cm[1:3]),
    carbon_stock_Mg_ha_upper100cm = sum(predicted_carbon_stock_per_10cm)
  ) %>%
  ungroup()

# #####
# Create summary sentences

summary_dataframe <- SG_modeling_dataframe %>%
  group_by(sample_ID) %>%
  slice(1)
summary_dataframe <- summary_dataframe %>% 
  select(-c("sediment_mean_depth_cm",
            "random_core_variable",
            "predicted_carbon_density",
            "predicted_carbon_stock_per_10cm"))

summary_sentences <- SG_modeling_dataframe %>%
  group_by(sample_ID) %>%
  slice(1) %>%  # take one row per sample
  mutate(summary = paste0(
    "For the seagrass bed at latitude ", latitude, " and longitude ", longitude, "\n",
    "The species identity ", seagrass_species, "\n",
    "The predicted carbon stock in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm,  digits=3, nsmall=3), " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm, digits=3, nsmall=3), " Mg/ha\n"
  )) %>%
  pull(summary)

# Save result
# -----
str_time <- format(Sys.time(),  "%Y%m%d_%H%M%S")
fname_result_csv <- paste(str_time, conf_file_result_csv, sep ="-")
fname_result_txt <- paste(str_time, conf_file_result_txt, sep ="-")
print(fname_result_csv)
print(fname_result_txt)

# file_result_csv <- paste(conf_tmp_directory_data, fname_result_csv, sep="/")
# file_result_txt <- paste(conf_tmp_directory_data, fname_result_txt, sep="/")

# if (conf_use_dummy_data) {
file_result_csv <- paste(conf_minio_user_local_root, conf_minio_user_bucket, fname_result_csv, sep="/")
file_result_txt <- paste(conf_minio_user_local_root, conf_minio_user_bucket, fname_result_txt, sep="/")

# summary_dataframe
# summary(summary_dataframe)
write.csv(summary_dataframe, file_result_csv)
# minio_status <- put_object(bucket=conf_minio_user_bucket, file=file_result_csv, object=paste(param_user_email, fcloud_result_csv, sep="/"))

# summary_sentences
# cat(summary_sentences, sep = "\n")
cat(summary_sentences, sep = "\n", file = file_result_txt)
# minio_status <- put_object(bucket=conf_minio_user_bucket, file=file_result_txt, object=paste(param_user_email, fcloud_result_txt, sep="/"))

print("Carbon predicted")

[1] "Carbon Density"
 Factor w/ 382 levels "Ærøya_Ærøya_NA_2017_8_7_58.417_8.763_3",..: 33 33 33 43 43 43 45 45 45 39 ...


[1] TRUE

character(0)

[1] TRUE

[1] "GAM_top_reduced_SGstock"
[1] "Carbon stock in the upper 30 and 100 cm"
[1] "20251212_105839-Seagrass_site_result.csv"
[1] "20251212_105839-Seagrass_site_result.txt"
[1] "Carbon predicted"
